# 카카오 4대 약관 RAG 평가기 (2단계 — 정교형)

문항, 정답 기준(gold), 팀별 답변 파일을 입력받아 0~100점을 매기는 평가기입니다.

**1단계 대비 변경점**
- 근거 채점: 문서명 정규화 + 조번호 파서(제7조/7/제7조의2 구분) + 부분점수 + Precision@4 블렌딩
  → 정답을 1순위에 두고 나머지 3자리를 무의미하게 채우는 전략에 벌점이 들어갑니다.
- 내용 채점: 어절 F1 대신 **LLM이 정답 핵심 내용을 항목별로 covered/missing/contradicted 판정**
  + **극성 게이트**(결론 방향이 반대면 강한 감점) + 수치·법조문 슬롯 정확도(정규식, 무료)
- LLM 채점: 문항 유형(극성/수치/열거/절차/일반) 자동 분류로 프롬프트 분기 + **3회 샘플링 중앙값**
  (자기 일관성) + 캐시 + 지수 백오프 재시도

토큰 사용량이 1단계보다 훨씬 많습니다(문항당 LLM 호출이 1+3=4회). 문항 수 × 팀 수 × 4회를
기준으로 예상 호출 수를 가늠하세요.

**셀은 반드시 위에서 아래로 순서대로 실행하세요.**

## 1. 설치

In [ ]:
import subprocess
import sys


def _pip_install(*packages: str) -> None:
    in_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix)
    cmd = [sys.executable, "-m", "pip", "install", "-q"]
    if not in_venv:
        cmd.append("--break-system-packages")
    cmd.extend(packages)
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)


_pip_install("pydantic>=2", "google-genai")
print("설치 완료")

## 2. API 키 주입

`GOOGLE_API_KEY`는 노트북 코드에 직접 쓰지 않습니다. Colab Secrets(왼쪽 열쇠 아이콘) 또는
로컬 환경변수(`export GOOGLE_API_KEY=...`)로 주입하세요.

In [ ]:
import os

try:
    from google.colab import userdata  # type: ignore
    key = userdata.get("GOOGLE_API_KEY")
    if key:
        os.environ.setdefault("GOOGLE_API_KEY", key)
except Exception:
    pass

if os.environ.get("GOOGLE_API_KEY"):
    print("GOOGLE_API_KEY 확인됨")
else:
    print("[경고] GOOGLE_API_KEY가 없습니다. 모든 LLM 채점이 실패/폴백 처리됩니다.")

## 3. 설정

In [ ]:
import os

GOLD_GLOB = os.environ.get("GOLD_GLOB", "gold_questions_public10*.json")
ANSWER_GLOB = os.environ.get("ANSWER_GLOB", "answers_public_*.json")
OUTPUT_PATH = os.environ.get("EVAL_OUTPUT_PATH", "eval_output.json")

# 개발 단계 공개 답변 파일은 blind_id가 아니라 team 필드("3" 등)만 가지고 있어
# BLIND01~05 형식이 아니다. 로컬/공개 테스트 시에만 이 값으로 강제 오버라이드한다.
# 실전(비공개 30문항)에서는 반드시 None으로 비워둔다.
DEV_BLIND_ID_OVERRIDE = os.environ.get("DEV_BLIND_ID_OVERRIDE")  # 예: "BLIND01"

# LLM 루브릭 채점 자기 일관성 샘플 수 (토큰을 더 쓰더라도 안정성을 높이고 싶으면 올린다)
RUBRIC_SAMPLE_COUNT = int(os.environ.get("RUBRIC_SAMPLE_COUNT", "3"))

# 정확한 모델 ID는 Google AI Studio / API 문서에서 최신값을 확인해서 채워 넣을 것.
GEMINI_MODEL_NAME = os.environ.get("GEMINI_MODEL_NAME", "gemini-3.5-flash")

print(f"GOLD_GLOB={GOLD_GLOB!r}")
print(f"ANSWER_GLOB={ANSWER_GLOB!r}")
print(f"OUTPUT_PATH={OUTPUT_PATH!r}")
print(f"DEV_BLIND_ID_OVERRIDE={DEV_BLIND_ID_OVERRIDE!r}")
print(f"RUBRIC_SAMPLE_COUNT={RUBRIC_SAMPLE_COUNT}")
print(f"GEMINI_MODEL_NAME={GEMINI_MODEL_NAME!r}")

## 4. 출력 계약 스키마

- 최상위 키는 `results`만 (`rank`, `schema_version` 금지)
- `total`은 반올림하지 않은 0~100 float, `failed`일 때만 `None`
- `status`는 `completed` / `partial` / `failed` 세 값만 허용

In [ ]:
from typing import Literal, Optional

from pydantic import BaseModel, ConfigDict, Field


class ResultItem(BaseModel):
    model_config = ConfigDict(extra="forbid")

    blind_id: str = Field(pattern=r"^BLIND\d{2}$")
    total: Optional[float]
    status: Literal["completed", "partial", "failed"]


class EvalOutput(BaseModel):
    model_config = ConfigDict(extra="forbid")

    results: list[ResultItem]

## 5. 골드셋 · 답변 파일 로더

In [ ]:
import glob
import json
import re
from dataclasses import dataclass
from typing import Any

_BLIND_ID_PATTERN = re.compile(r"^BLIND\d{2}$")
_BLIND_ID_SEARCH_PATTERN = re.compile(r"BLIND\d{2}")


@dataclass(frozen=True)
class GoldQuestion:
    id: str
    question: str
    gold_articles: list[dict[str, Any]]
    key_facts: list[str]


@dataclass(frozen=True)
class AnswerItem:
    qid: str
    retrieved: list[list[Any]]
    answer: str


@dataclass(frozen=True)
class AnswerFile:
    blind_id: str
    path: str
    answers: dict[str, AnswerItem]


class LoaderError(Exception):
    """입력 파일 검증 실패. 이 예외를 잡은 상위 로직은 해당 팀을 failed 처리한다."""


def load_gold_questions(path: str) -> list[GoldQuestion]:
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)

    questions = raw.get("questions")
    if not isinstance(questions, list) or not questions:
        raise LoaderError(f"골드셋에 questions 배열이 없거나 비어 있습니다: {path}")

    result: list[GoldQuestion] = []
    seen_ids: set[str] = set()
    for q in questions:
        qid = q.get("id")
        if not qid:
            raise LoaderError(f"gold 문항에 id가 없습니다: {q}")
        if qid in seen_ids:
            raise LoaderError(f"gold 문항 id가 중복되었습니다: {qid}")
        seen_ids.add(qid)

        gold_articles = q.get("gold_articles")
        if not isinstance(gold_articles, list) or not gold_articles:
            raise LoaderError(f"{qid}: gold_articles가 없거나 비어 있습니다")

        key_facts = q.get("key_facts")
        if not isinstance(key_facts, list) or not key_facts:
            raise LoaderError(f"{qid}: key_facts가 없거나 비어 있습니다")

        result.append(
            GoldQuestion(
                id=qid,
                question=q.get("question", ""),
                gold_articles=gold_articles,
                key_facts=key_facts,
            )
        )
    return result


def load_answer_file(path: str, blind_id_override: Optional[str] = None) -> AnswerFile:
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)

    blind_id = blind_id_override or raw.get("blind_id") or raw.get("team")
    if not blind_id:
        m = _BLIND_ID_SEARCH_PATTERN.search(path.upper())
        if m:
            blind_id = m.group(0)
    if not blind_id:
        raise LoaderError(f"blind_id를 파일에서도, 파일명에서도, 인자로도 찾을 수 없습니다: {path}")

    answers_raw = raw.get("answers")
    if not isinstance(answers_raw, list):
        raise LoaderError(f"{path}: answers 배열이 없습니다")

    answers: dict[str, AnswerItem] = {}
    for a in answers_raw:
        qid = a.get("qid")
        if not qid:
            continue
        if qid in answers:
            raise LoaderError(f"{path}: qid가 중복되었습니다: {qid}")
        answers[qid] = AnswerItem(
            qid=qid,
            retrieved=a.get("retrieved", []) or [],
            answer=a.get("answer", "") or "",
        )

    return AnswerFile(blind_id=str(blind_id), path=path, answers=answers)


def discover_answer_files(pattern: str) -> list[str]:
    paths = sorted(glob.glob(pattern))
    if not paths:
        raise LoaderError(f"패턴에 해당하는 파일이 없습니다: {pattern}")
    return paths


def is_valid_blind_id(blind_id: str) -> bool:
    return bool(_BLIND_ID_PATTERN.match(blind_id))


def validate_blind_ids(answer_files: list[AnswerFile]) -> None:
    seen: set[str] = set()
    for af in answer_files:
        if not _BLIND_ID_PATTERN.match(af.blind_id):
            raise LoaderError(f"blind_id 형식이 올바르지 않습니다: {af.blind_id} ({af.path})")
        if af.blind_id in seen:
            raise LoaderError(f"blind_id가 중복되었습니다: {af.blind_id}")
        seen.add(af.blind_id)

## 6. 정규화 · 분류 유틸

근거 채점과 LLM 채점이 공통으로 쓰는 순수 함수들입니다. API 호출이 없어 토큰 비용이 들지
않고, 그만큼 정확도를 최대한 끌어올려야 하는 부분입니다.

In [ ]:
from typing import NamedTuple

# ---- 문서명 정규화 ----
_CANON_DOC_NAMES = {
    "카카오계정약관": "카카오계정 약관",
    "카카오위치정보이용약관": "카카오 위치정보 이용약관",
    "카카오통합서비스약관": "카카오 통합서비스약관",
    "카카오통합약관": "카카오 통합 약관",
}


def canon_doc(name) -> Optional[str]:
    if name is None:
        return None
    key = re.sub(r"\s+", "", str(name)).replace("(", "").replace(")", "")
    return _CANON_DOC_NAMES.get(key)


# ---- 조번호 파서 ----
class ArticleRef(NamedTuple):
    jo: int
    ui: int = 0
    hang: Optional[int] = None


_JO = re.compile(r"제?\s*(\d+)\s*조(?:\s*의\s*(\d+))?(?:\s*제?\s*(\d+)\s*항)?")


def parse_article(raw) -> Optional[ArticleRef]:
    if isinstance(raw, bool):
        return None
    if isinstance(raw, int):
        return ArticleRef(raw, 0, None)
    if isinstance(raw, float) and raw.is_integer():
        return ArticleRef(int(raw), 0, None)
    if not isinstance(raw, str):
        return None
    s = raw.strip()
    if not s:
        return None
    if s.isdigit():
        return ArticleRef(int(s), 0, None)
    m = _JO.search(s)
    if not m:
        return None
    return ArticleRef(
        int(m.group(1)),
        int(m.group(2)) if m.group(2) else 0,
        int(m.group(3)) if m.group(3) else None,
    )


def match_score(gold_doc, gold_article, pred_doc, pred_article) -> float:
    """1.0=완전일치, 0.3=같은 조 다른 항(의N), 0.0=불일치. gold보다 더 구체적(항까지 지정)이면 1.0 유지."""
    if canon_doc(gold_doc) != canon_doc(pred_doc) or canon_doc(gold_doc) is None:
        return 0.0
    ga, pa = parse_article(gold_article), parse_article(pred_article)
    if ga is None or pa is None:
        return 0.0
    if (ga.jo, ga.ui) == (pa.jo, pa.ui):
        return 1.0
    if ga.jo == pa.jo and ga.ui != pa.ui:
        return 0.3
    return 0.0


# ---- 문항 유형 분류 (ptype 필드에 의존하지 않음 — 비공개셋엔 없을 수 있음) ----
def classify_type(question: str, key_facts: list[str]) -> str:
    first_fact = key_facts[0].strip() if key_facts else ""
    if re.search(r"(나요|인가요|습니까)\s*[?？]?\s*$", question.strip()) and re.match(
        r"^(아니오|아니요|않|없|아니|금지)", first_fact
    ):
        return "polarity"
    combined = question + " " + " ".join(key_facts)
    if re.search(r"(\d+\s*(세|일|개월|년|시간|%)|제\s*\d+\s*조)", combined):
        return "numeric"
    if re.search(r"\d+\s*가지", question) or len(key_facts) >= 4:
        return "enumerate"
    if re.search(r"(순서로|먼저|차례로)", question):
        return "procedure"
    return "basic"


# ---- 수치·법조문 슬롯 ----
_SLOT_PAT = re.compile(
    r"(제\s*\d+\s*조(?:\s*의\s*\d+)?(?:\s*제\s*\d+\s*항)?|\d+\s*(?:세|일|개월|년|시간|%|인|명|가지))"
)


def extract_slots(text: str) -> set[str]:
    return {re.sub(r"\s+", "", m[0] if isinstance(m, tuple) else m) for m in _SLOT_PAT.findall(text)}


def slot_recall(gold_text: str, answer_text: str):
    """반환: (recall, missing_list) 또는 슬롯이 없으면 None."""
    gold_slots = extract_slots(gold_text)
    if not gold_slots:
        return None
    ans_slots = extract_slots(answer_text)
    missing = sorted(gold_slots - ans_slots)
    recall = (len(gold_slots) - len(missing)) / len(gold_slots)
    return recall, missing

## 7. 채점 — 근거 조항 (graded MRR + Precision@4, 0~1 정규화)

any-match(gold가 여러 개여도 하나만 맞히면 그 순위로 인정) 방식은 유지하되, 이번에는
- 부분점수: 같은 조인데 `의N`만 다르면 0.3점
- Precision@4: 상위 4개 중 완전일치 비율을 15% 반영 → 정답을 1순위에 놓고 나머지를
  무의미하게 채우는 전략에 실제로 벌점이 들어갑니다.

In [ ]:
def score_evidence(gold_articles: list[dict[str, Any]], retrieved: list[list[Any]]) -> float:
    if not retrieved:
        return 0.0

    graded_mrr = 0.0
    exact_hits = 0
    k = min(4, len(retrieved))

    for rank, item in enumerate(retrieved[:4], start=1):
        if len(item) < 2:
            continue
        doc, article = item[0], item[1]
        best_match = max(
            (match_score(g.get("doc"), g.get("article"), doc, article) for g in gold_articles),
            default=0.0,
        )
        if best_match > 0:
            graded_mrr = max(graded_mrr, best_match / rank)
        if best_match >= 1.0:
            exact_hits += 1

    precision = exact_hits / k if k > 0 else 0.0
    score = 0.85 * graded_mrr + 0.15 * precision
    return max(0.0, min(1.0, score))

## 8. 공통 — Gemini 호출 유틸

내용 채점(9번)과 LLM 루브릭 채점(10번)이 공유하는 API 호출/재시도/캐시 로직입니다.
SDK는 신규 `google-genai`를 사용합니다(구 `google-generativeai`는 지원 종료됨).

In [ ]:
import hashlib
import time

_CACHE: dict[str, Any] = {}


def cache_key(kind: str, blind_id: str, qid: str, prompt_version: str, prompt: str) -> str:
    raw = f"{kind}|{blind_id}|{qid}|{prompt_version}|{prompt}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


def call_gemini(prompt: str) -> str:
    from google import genai
    from google.genai import types

    api_key = os.environ.get("GOOGLE_API_KEY")
    if not api_key:
        raise RuntimeError("GOOGLE_API_KEY 환경변수가 설정되어 있지 않습니다. 2번 셀을 먼저 확인하세요.")

    client = genai.Client(api_key=api_key)
    response = client.models.generate_content(
        model=GEMINI_MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.0, response_mime_type="application/json"),
    )
    return response.text


def call_gemini_with_retry(prompt: str, max_retries: int = 3) -> str:
    """일시적 오류(레이트리밋 등)만 재시도한다. API 키 미설정은 설정 오류이므로 즉시 던진다."""
    last_exc: Optional[Exception] = None
    for attempt in range(max_retries):
        try:
            return call_gemini(prompt)
        except RuntimeError:
            raise
        except Exception as exc:  # noqa: BLE001
            last_exc = exc
            time.sleep(2**attempt)
    raise last_exc  # type: ignore[misc]


def strip_code_fence(raw: str) -> str:
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        cleaned = cleaned.split("\n", 1)[-1] if "\n" in cleaned else cleaned
    return cleaned

## 9. 채점 — 답변 내용 (LLM 키팩트 판정 + 극성 게이트 + 슬롯, 0~1 정규화)

`detail.degraded == True`이면 LLM 판정이 실패해 결정론적 F1로 대체되었다는 뜻이며,
이 경우 해당 문항은 11번 셀에서 `partial` 취급됩니다(내용 채점도 정밀 경로가 아니었으므로).

In [ ]:
from collections import Counter

CONTENT_PROMPT_VERSION = "content_v2"


def score_content(
    question: str,
    key_facts: list[str],
    answer: str,
    blind_id: str = "",
    qid: str = "",
) -> dict:
    slot_text = " ".join(key_facts)
    slot_result = slot_recall(slot_text, answer)
    if slot_result is not None:
        recall, missing_slots = slot_result
        slot_multiplier = 0.7 + 0.3 * recall
    else:
        slot_multiplier = 1.0
        missing_slots = []

    try:
        nugget = _nugget_verify(question, key_facts, answer, blind_id, qid)
        facts = nugget["facts"]
        contradicted = sum(1 for f in facts if f["status"] == "contradicted")
        covered = sum(1 for f in facts if f["status"] == "covered")
        coverage = covered / len(key_facts) if key_facts else 0.0
        polarity_match = nugget.get("polarity_match")

        score = coverage
        if polarity_match is False:
            score *= 0.15
        if contradicted > 0:
            score *= 0.5
        score *= slot_multiplier
        score = max(0.0, min(1.0, score))

        return {
            "score": score,
            "detail": {
                "method": "llm_nugget",
                "degraded": False,
                "coverage": coverage,
                "contradicted": contradicted,
                "polarity_match": polarity_match,
                "missing_slots": missing_slots,
                "slot_multiplier": slot_multiplier,
            },
        }
    except Exception as exc:  # noqa: BLE001 — 내용 채점은 절대 예외를 던지지 않는다
        print(f"[알림] {blind_id} {qid} 내용 채점 LLM 실패, 결정론적 F1로 대체: {exc}")
        f1 = _word_f1(key_facts, answer) * slot_multiplier
        f1 = max(0.0, min(1.0, f1))
        return {
            "score": f1,
            "detail": {
                "method": "fallback_f1",
                "degraded": True,
                "llm_error": str(exc),
                "missing_slots": missing_slots,
                "slot_multiplier": slot_multiplier,
            },
        }


def _word_f1(key_facts: list[str], answer: str) -> float:
    gold_tokens = Counter(" ".join(key_facts).split())
    pred_tokens = Counter(answer.split())
    overlap = sum((gold_tokens & pred_tokens).values())
    if overlap == 0:
        return 0.0
    precision = overlap / sum(pred_tokens.values())
    recall = overlap / sum(gold_tokens.values())
    return 2 * precision * recall / (precision + recall)


def _nugget_verify(question: str, key_facts: list[str], answer: str, blind_id: str, qid: str) -> dict:
    prompt = _build_nugget_prompt(question, key_facts, answer)
    key = cache_key("nugget", blind_id, qid, CONTENT_PROMPT_VERSION, prompt)
    if key in _CACHE:
        return _CACHE[key]

    raw = call_gemini_with_retry(prompt)
    result = _parse_nugget_response(strip_code_fence(raw), len(key_facts))
    _CACHE[key] = result
    return result


def _build_nugget_prompt(question: str, key_facts: list[str], answer: str) -> str:
    numbered_facts = "\n".join(f"{i + 1}. {f}" for i, f in enumerate(key_facts))
    return f"""당신은 카카오 약관 기반 RAG 답변을 채점하는 평가자입니다.

[질문]
{question}

[정답 핵심 내용] (번호 순서대로)
{numbered_facts}

[참가팀 답변]
{answer}

각 핵심 내용이 답변에 실질적으로 포함되어 있는지 판정하세요.
- "covered": 답변에 그 내용이 명시적으로 또는 동등한 의미로 포함됨
- "missing": 답변에 없음
- "contradicted": 답변이 그 내용과 반대되는 주장을 함

또한 질문이 예/아니오 또는 긍정/부정을 묻는 질문이라면, 답변의 최종 결론이 정답의
결론과 같은 방향인지("polarity_match": true) 반대 방향인지("polarity_match": false)를
판정하세요. 회피성 서술("경우에 따라 다르다" 등)도 결론 불일치로 간주하세요.
방향성이 없는 질문(단순 설명·나열형)이면 "polarity_match": null로 답하세요.

다음 JSON만 출력하세요. 다른 말은 쓰지 마세요.
{{"facts": [{{"index": 1, "status": "covered"}}, ...], "polarity_match": true, "polarity_reason": "<40자 이내>"}}"""


def _parse_nugget_response(cleaned: str, num_facts: int) -> dict:
    data = json.loads(cleaned)
    facts = data.get("facts")
    if not isinstance(facts, list) or len(facts) != num_facts:
        raise ValueError(f"facts 배열 길이가 key_facts와 다릅니다: {data}")
    for f in facts:
        if f.get("status") not in ("covered", "missing", "contradicted"):
            raise ValueError(f"알 수 없는 status 값: {f}")

    polarity_match = data.get("polarity_match", None)
    if polarity_match not in (True, False, None):
        raise ValueError(f"polarity_match 값이 올바르지 않습니다: {polarity_match}")

    return {"facts": facts, "polarity_match": polarity_match}

## 10. 채점 — LLM 루브릭 (정확성·근거성·완결성·명료성, 3회 샘플링 중앙값)

문항 유형에 따라 프롬프트의 "특히 확인할 것" 절이 자동으로 바뀝니다. temperature 0에서도
API 응답은 완전히 결정론적이지 않으므로, `RUBRIC_SAMPLE_COUNT`회 호출해 중앙값을 취합니다.

In [ ]:
import statistics

RUBRIC_PROMPT_VERSION = "rubric_v2"

_TYPE_HINTS = {
    "polarity": (
        "이 문항은 결론의 방향(긍정/부정)이 정답과 일치하는지가 핵심입니다. "
        "결론이 반대로 서술되었다면 정확성과 근거성을 0~1점으로 처리하세요. "
        "회피성 서술('경우에 따라 다르다' 등)도 결론 불일치로 간주하세요."
    ),
    "numeric": (
        "이 문항은 정확한 수치·기간·법조문 번호가 핵심입니다. "
        "정답의 숫자나 조번호와 다른 값을 답변이 제시했다면 정확성을 낮게 주세요."
    ),
    "enumerate": (
        "이 문항은 여러 항목을 빠짐없이 나열·설명해야 합니다. "
        "이름만 나열하고 설명이 없는 답변은 완결성을 3점 이상 주지 마세요."
    ),
    "procedure": (
        "이 문항은 절차의 순서가 핵심입니다. "
        "항목은 다 언급했지만 순서가 뒤바뀐 답변은 정확성을 낮게 주세요."
    ),
    "basic": "",
}


def score_llm(
    question: str,
    gold_articles: list[dict[str, Any]],
    key_facts: list[str],
    answer: str,
    blind_id: str = "",
    qid: str = "",
) -> dict:
    """실패 시 예외를 던진다 — 상위(run)에서 잡아 partial 처리."""
    qtype = classify_type(question, key_facts)
    prompt = _build_rubric_prompt(question, gold_articles, key_facts, answer, qtype)

    key = cache_key("rubric", blind_id, qid, RUBRIC_PROMPT_VERSION, prompt)
    if key in _CACHE:
        return _CACHE[key]

    samples = [
        _parse_rubric_response(strip_code_fence(call_gemini_with_retry(prompt)))
        for _ in range(RUBRIC_SAMPLE_COUNT)
    ]
    result = _median_aggregate(samples)
    _CACHE[key] = result
    return result


def _build_rubric_prompt(question, gold_articles, key_facts, answer, qtype) -> str:
    citation_lines = "\n".join(f"- {g.get('citation') or g.get('doc')}" for g in gold_articles)
    facts_lines = "\n".join(f"{i + 1}. {f}" for i, f in enumerate(key_facts))
    hint = _TYPE_HINTS.get(qtype, "")
    hint_block = f"\n[특히 확인할 것]\n{hint}\n" if hint else ""

    return f"""당신은 카카오 약관 기반 RAG 답변을 채점하는 평가자입니다.
아래 질문에 대한 참가팀의 답변을 정답과 비교하여 4개 항목을 평가하세요.

[질문]
{question}

[근거 조항]
{citation_lines}

[정답 핵심 내용]
{facts_lines}

[참가팀 답변]
{answer}
{hint_block}
[평가 항목 — 각 0~4점]
정확성(accuracy): 답변의 사실 주장이 정답과 일치하는가.
근거성(groundedness): 답변 내용이 근거 조항 범위 내에 있는가(지어낸 내용이 없는가).
완결성(completeness): 정답 핵심 내용을 빠짐없이 다루었는가.
명료성(clarity): 구조와 표현이 읽기 쉬운가.

다음 JSON만 출력하세요. 다른 말은 쓰지 마세요.
{{"accuracy": <0-4>, "groundedness": <0-4>, "completeness": <0-4>, "clarity": <0-4>, "reason": "<40자 이내>"}}"""


def _parse_rubric_response(cleaned: str) -> dict:
    data = json.loads(cleaned)
    required = ("accuracy", "groundedness", "completeness", "clarity")
    for key in required:
        if key not in data:
            raise ValueError(f"LLM 응답에 {key} 필드가 없습니다: {data}")
        if not (0 <= data[key] <= 4):
            raise ValueError(f"LLM 응답 {key} 값이 범위를 벗어났습니다: {data[key]}")
    data.setdefault("reason", "")
    return data


def _median_aggregate(samples: list[dict]) -> dict:
    agg = {
        k: statistics.median(s[k] for s in samples)
        for k in ("accuracy", "groundedness", "completeness", "clarity")
    }
    agg["reason"] = samples[0].get("reason", "")
    agg["n_samples"] = len(samples)
    return agg

## 11. 집계 — 문항 단위 + 팀 단위

배점: 근거(MRR) 20 + 내용(F1) 30 + LLM 50.

- 결측 문항 -> 0점 처리, 평균 분모에 포함 (역인센티브 방지)
- LLM 루브릭 실패 -> 근거+내용(50점 만점) 재정규화, `partial`
- 내용 채점이 LLM 실패로 폴백(`degraded`)된 경우도 `partial` — 정밀 경로가 아니었으므로
  다른 문항과 동일한 기준으로 채점됐다고 보기 어렵기 때문

In [ ]:
EVIDENCE_WEIGHT = 20.0
CONTENT_WEIGHT = 30.0
LLM_WEIGHT = 50.0

_LLM_SUBWEIGHTS = {
    "accuracy": 0.40,
    "groundedness": 0.25,
    "completeness": 0.20,
    "clarity": 0.15,
}


@dataclass
class QuestionScore:
    qid: str
    score_0_100: float
    llm_failed: bool


def llm_subscore(llm_result: dict) -> float:
    weighted = sum(_LLM_SUBWEIGHTS[key] * llm_result[key] for key in _LLM_SUBWEIGHTS)
    return weighted / 4.0


def aggregate_question(
    qid: str,
    evidence_score: float,
    content_result: dict,
    llm_result: Optional[dict],
) -> QuestionScore:
    content_score = content_result["score"]
    content_degraded = content_result["detail"].get("degraded", False)

    if llm_result is not None:
        earned = (
            evidence_score * EVIDENCE_WEIGHT
            + content_score * CONTENT_WEIGHT
            + llm_subscore(llm_result) * LLM_WEIGHT
        )
        return QuestionScore(qid=qid, score_0_100=earned, llm_failed=content_degraded)

    earned_without_llm = evidence_score * EVIDENCE_WEIGHT + content_score * CONTENT_WEIGHT
    max_without_llm = EVIDENCE_WEIGHT + CONTENT_WEIGHT
    renormalized = (earned_without_llm / max_without_llm) * 100.0
    return QuestionScore(qid=qid, score_0_100=renormalized, llm_failed=True)


def aggregate_team(question_scores: list[QuestionScore]) -> tuple[Optional[float], str]:
    if not question_scores:
        return None, "failed"

    total = sum(qs.score_0_100 for qs in question_scores) / len(question_scores)
    status = "partial" if any(qs.llm_failed for qs in question_scores) else "completed"
    return total, status

## 12. 저장 + 재검증

In [ ]:
def save_eval_output(path: str, results: list[ResultItem]) -> None:
    output = EvalOutput(results=results)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(output.model_dump(), f, ensure_ascii=False, indent=2)

    _revalidate(path)


def _revalidate(path: str) -> None:
    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    if set(data.keys()) != {"results"}:
        raise AssertionError(f"최상위 키가 results만이어야 합니다. 실제: {sorted(data.keys())}")

    EvalOutput(**data)

## 13. 실행

골드셋과 답변 파일들을 찾아 팀별로 채점하고 저장합니다. LLM 호출 실패는 문항 단위로
격리해 `partial`로 흡수하며, 이 셀 자체는 예외를 던지지 않습니다.

In [ ]:
def evaluate_team(gold_questions: list[GoldQuestion], answer_file: AnswerFile) -> ResultItem:
    question_scores: list[QuestionScore] = []

    for gq in gold_questions:
        answer_item = answer_file.answers.get(gq.id)
        retrieved = answer_item.retrieved if answer_item else []
        answer_text = answer_item.answer if answer_item else ""

        evidence_score = score_evidence(gq.gold_articles, retrieved)

        try:
            content_result = score_content(
                gq.question, gq.key_facts, answer_text, answer_file.blind_id, gq.id
            )
        except Exception as exc:  # noqa: BLE001 — score_content는 자체 폴백을 갖지만 방어적으로 한 번 더
            print(f"[경고] {answer_file.blind_id} {gq.id} 내용 채점 완전 실패, 0점 처리: {exc}")
            content_result = {"score": 0.0, "detail": {"degraded": True, "error": str(exc)}}

        llm_result = None
        try:
            llm_result = score_llm(
                question=gq.question,
                gold_articles=gq.gold_articles,
                key_facts=gq.key_facts,
                answer=answer_text,
                blind_id=answer_file.blind_id,
                qid=gq.id,
            )
        except Exception as exc:  # noqa: BLE001 — 문항 단위 실패 격리가 목적
            print(f"[경고] {answer_file.blind_id} {gq.id} LLM 루브릭 채점 실패: {exc}")

        question_scores.append(
            aggregate_question(gq.id, evidence_score, content_result, llm_result)
        )

    total, status = aggregate_team(question_scores)
    return ResultItem(blind_id=answer_file.blind_id, total=total, status=status)


def run() -> None:
    gold_paths = discover_answer_files(GOLD_GLOB)
    gold_questions = load_gold_questions(gold_paths[0])
    print(f"골드셋 로드: {gold_paths[0]} ({len(gold_questions)}문항)")

    answer_paths = discover_answer_files(ANSWER_GLOB)
    print(f"답변 파일 {len(answer_paths)}개 발견")
    print(f"예상 LLM 호출 수(대략): 문항수 x 팀수 x (1 + {RUBRIC_SAMPLE_COUNT})")

    results: list[ResultItem] = []
    for path in answer_paths:
        try:
            answer_file = load_answer_file(path, blind_id_override=DEV_BLIND_ID_OVERRIDE)
        except LoaderError as exc:
            print(f"[실패] {path} 로딩 실패: {exc}")
            continue

        if not is_valid_blind_id(answer_file.blind_id):
            print(
                f"[스킵] {path}: blind_id '{answer_file.blind_id}'가 BLIND01~05 형식이 "
                "아닙니다. 실전 파일이 아니라면 3번 셀의 DEV_BLIND_ID_OVERRIDE를 설정하세요."
            )
            continue

        try:
            result = evaluate_team(gold_questions, answer_file)
        except Exception as exc:  # noqa: BLE001 — 팀 단위 완전 실패
            print(f"[실패] {answer_file.blind_id} 채점 중단: {exc}")
            result = ResultItem(blind_id=answer_file.blind_id, total=None, status="failed")

        results.append(result)
        print(f"  {result.blind_id}: total={result.total} status={result.status}")

    save_eval_output(OUTPUT_PATH, results)
    print(f"저장 완료: {OUTPUT_PATH}")


run()